[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-multiple-reg.ipynb)

# Multiple Regression

*AIBits Academy · Machine Learning End To End · Supervised Learning*

Predicting outcomes from several input variables simultaneously — the workhorse of analytical modelling in finance, operations, and healthcare.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **🎯 Intuition First**
>
> One predictor fits a line. Two predictors fit a tilted **plane** hovering over a floor — length on one edge, width on the other, price as the height. Three or more fit a "hyperplane" you can't picture — but here's the reassuring part: **the mathematics is identical at every dimension**. Nothing new to learn; just more columns in the same equation.

> **📋 Real-World Case Study — Bike-Sharing Demand Prediction**
>
> A city bike-share operator needs to predict hourly rental demand from weather (temperature, humidity, windspeed), day type (weekday/weekend/holiday), and season — a textbook multiple regression setup with several simultaneous numeric and categorical predictors. (Forecasting demand specifically as a sequence *over time* is a distinct time-series problem, covered in a separate specialised track — here we're treating each hour as an independent row of features, the multiple-regression framing.)

## From Simple to Multiple

Simple linear regression uses one predictor. Multiple linear regression uses p predictors:

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_p x_p = \mathbf{x}^{\top}\theta$$

In matrix form: ŷ = **Xθ** where **X** ∈ ℝⁿˣ⁽ᵖ⁺¹⁾ (with a ones column for bias). The Normal Equation and gradient descent generalise directly.

## Indian Real Estate Example — Ahmedabad Apartments

Predict apartment sale price (₹ lakhs) from area (sq ft), floors, age (years), and distance to BRTS (km):

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# Ahmedabad apartment dataset (synthetic but representative)
np.random.seed(42)
n = 500
area   = np.random.randint(500, 2500, n)       # sq ft
floors = np.random.randint(1, 25, n)
age    = np.random.randint(0, 30, n)             # years old
dist   = np.round(np.random.uniform(0.2, 8, n), 1)  # km to BRTS
price  = (0.08*area + 1.5*floors - 1.2*age
          - 3.5*dist + np.random.normal(0, 8, n) + 40)

df = pd.DataFrame({
    'area': area, 'floors': floors,
    'age': age, 'dist_brts': dist, 'price_lakh': price
})

X = df[['area','floors','age','dist_brts']]
y = df['price_lakh']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

for name, model in [('OLS', LinearRegression()),
                    ('Ridge α=1', Ridge(1)),
                    ('Lasso α=0.5', Lasso(0.5))]:
    model.fit(X_tr_s, y_tr)
    y_hat = model.predict(X_te_s)
    print(f"{name:12s}  R²={r2_score(y_te, y_hat):.4f}  MAE={mean_absolute_error(y_te, y_hat):.1f} lakhs")

# Interpret OLS coefficients
ols = LinearRegression().fit(X_tr_s, y_tr)
for feat, coef in zip(X.columns, ols.coef_):
    print(f"  {feat:12s}: {coef:+.2f} ₹ lakhs per SD increase")

## Multicollinearity — Detection and Fix

Multicollinearity (high correlation among predictors) inflates standard errors, making individual coefficients unreliable. Variance Inflation Factor (VIF) detects it:

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

Xs = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
vif = pd.DataFrame({
    'feature': Xs.columns,
    'VIF': [variance_inflation_factor(Xs.values, i) for i in range(Xs.shape[1])]
})
print(vif.to_string(index=False))
# VIF > 10 → severe multicollinearity → use Ridge or drop feature

## Interpreting Coefficients — Standardised vs Raw

**Raw coefficients** tell you the change in ŷ for a 1-unit increase in xⱼ, holding others constant. Units differ, so magnitudes aren't comparable. 
 
**Standardised coefficients (beta coefficients)** — fit on scaled features — tell you which feature has the largest *relative* impact. In the Ahmedabad example, area has the largest effect per standard deviation.

## Try It — Watch VIF and Coefficients as Correlation Rises

Two standardised predictors, correlation ρ tunable by slider, with a fixed true relationship y = x₁ + x₂ + noise underneath. Drag ρ toward 1, then click "Resample noise" a few times at each setting — at low ρ the coefficients barely move between samples; at high ρ they swing wildly (even flipping sign) despite R² staying almost unchanged. That gap between "prediction is fine" and "coefficients are unreliable" is exactly what VIF flags.

> **💡 Going Deeper**
>
> The Ridge/Lasso calls above only scratch the surface. The next chapter, **Regularization**, derives the Ridge closed-form solution, explains geometrically why Lasso zeroes coefficients and Ridge doesn't, and works through ElasticNet — essential for any dataset with more than a handful of correlated features.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Two features, one model

`price = 10 + 0.5·area + 4·rooms` exactly. Fit a `LinearRegression` and store the two coefficients (area first) in `coefs` and the intercept in `b0`.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
area = np.array([50, 70, 90, 60, 120, 80.0]); rooms = np.array([1, 2, 3, 2, 4, 2.0])
price = 10 + 0.5 * area + 4 * rooms
X = np.column_stack([area, rooms])
coefs = b0 = None   # TODO


In [ ]:
try:
    check("coefficients", np.allclose(coefs, [0.5, 4]))
    check("intercept", abs(b0 - 10) < 1e-6)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression
area = np.array([50, 70, 90, 60, 120, 80.0]); rooms = np.array([1, 2, 3, 2, 4, 2.0])
price = 10 + 0.5 * area + 4 * rooms
X = np.column_stack([area, rooms])
m = LinearRegression().fit(X, price)
coefs, b0 = m.coef_, m.intercept_

```

</details>

### Exercise 2 · Medium · Adjusted R²

Adding features can only raise R², so also report adjusted R². Write `adj_r2(r2, n, p)` = `1 - (1 - r2) * (n - 1) / (n - p - 1)`.

In [ ]:
def adj_r2(r2, n, p):
    pass   # TODO


In [ ]:
try:
    check("example 1", round(adj_r2(0.90, 100, 5), 4) == 0.8947)
    check("adjusted is never above R2", adj_r2(0.8, 50, 10) < 0.8)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def adj_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

```

</details>

### Exercise 3 · Stretch · Spot multicollinearity with VIF

`x2` is almost a copy of `x1`. Compute the variance inflation factor of every column with `variance_inflation_factor` into a dict `vif`, and store the column with the highest VIF in `worst`. (VIF > 10 is a red flag.)

In [ ]:
import numpy as np, pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
rng = np.random.default_rng(4)
x1 = rng.normal(size=200)
X = pd.DataFrame({"x1": x1, "x2": x1 + rng.normal(scale=0.05, size=200), "x3": rng.normal(size=200)})
vif = {}
worst = None   # TODO


In [ ]:
try:
    check("three VIFs", len(vif) == 3)
    check("x1/x2 are flagged", vif["x1"] > 10 and vif["x2"] > 10 and vif["x3"] < 5)
    check("worst is x1 or x2", worst in ("x1", "x2"))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np, pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
rng = np.random.default_rng(4)
x1 = rng.normal(size=200)
X = pd.DataFrame({"x1": x1, "x2": x1 + rng.normal(scale=0.05, size=200), "x3": rng.normal(size=200)})
vif = {c: variance_inflation_factor(X.values, i) for i, c in enumerate(X.columns)}
worst = max(vif, key=vif.get)

```

Dropping one of the near-duplicate columns (or using Ridge) stabilises the coefficients.

</details>

---
*Back to the course: **Machine Learning End To End → Multiple Regression**.*